In [1]:
!pip install -r ../requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [ ]:
%cd ../
import torch
import numpy as np
import torch
from PIL import Image

from models.yolo import Model


/home/su_kark/Documents/Master Research Project/quantized-yolov5


/home/su_kark/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
ckpt = torch.load('./runs/train/exp/weights/best.pt', map_location='cpu')

print(len(ckpt))
for k, v in ckpt.items():
    print(k)
    if 'scaling' in k:
        print(v)


67
model.0.conv.weight
model.0.bn.weight
model.0.bn.bias
model.0.bn.running_mean
model.0.bn.running_var
model.0.bn.num_batches_tracked
model.0.act.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
tensor(2.58594, dtype=torch.float16)
model.2.conv.weight
model.2.bn.weight
model.2.bn.bias
model.2.bn.running_mean
model.2.bn.running_var
model.2.bn.num_batches_tracked
model.2.act.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
tensor(2.58594, dtype=torch.float16)
model.4.conv.weight
model.4.bn.weight
model.4.bn.bias
model.4.bn.running_mean
model.4.bn.running_var
model.4.bn.num_batches_tracked
model.4.act.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
tensor(2.58594, dtype=torch.float16)
model.6.conv.weight
model.6.bn.weight
model.6.bn.bias
model.6.bn.running_mean
model.6.bn.running_var
model.6.bn.num_batches_tracked
model.6.act.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
tensor(2.58594, dtype=torc

/tmp/ipykernel_2563109/1623036458.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load('./runs/train/exp/weights/best.pt', map_location='cpu')


In [4]:
import cv2
from utils.utils import letterbox

image = "/home/su_kark/Documents/Master Research Project/datasets/hpd2_416_augmented/test/images/img_0107_jpg.rf.fe620b41ce8f044692281de1021cb868.jpg"

img_org = cv2.imread(image)
img = img_org.copy()

h, w, _ = img_org.shape
img, ratio, (dw, dh) = letterbox(img, (416,416), auto=False)

img = img[:, :, ::-1]

img_array = np.array(img, dtype=np.float32)  # (H, W, C)

# Rearrange to (C, H, W)
img_array = np.transpose(img_array, (2, 0, 1))

# Add batch dimension → (B, C, H, W)
img_array = np.expand_dims(img_array, axis=0)
output_path = "input_image.npy"

# Save as .npy
np.save(output_path, img_array)

print(img.shape)
print(f"Saved image as {output_path} with shape {img_array.shape}")


(416, 416, 3)
Saved image as input_image.npy with shape (1, 3, 416, 416)


In [5]:
test_input_np = np.load("input_image.npy")
test_input = torch.from_numpy(test_input_np).float() / 255.0

In [6]:
print('INPUT:', test_input.shape)
print(test_input)

INPUT: torch.Size([1, 3, 416, 416])
tensor([[[[0.19608, 0.13333, 0.11373,  ..., 0.89020, 0.90980, 0.89804],
          [0.36471, 0.26667, 0.23529,  ..., 0.27843, 0.31765, 0.32549],
          [0.26275, 0.21176, 0.28627,  ..., 0.09412, 0.12941, 0.15294],
          ...,
          [0.01176, 0.01176, 0.01176,  ..., 0.18039, 0.12549, 0.07451],
          [0.01176, 0.01176, 0.01176,  ..., 0.14902, 0.09020, 0.03529],
          [0.01176, 0.01176, 0.01176,  ..., 0.12549, 0.06275, 0.00784]],

         [[0.19608, 0.13333, 0.11373,  ..., 0.89020, 0.90980, 0.89804],
          [0.36471, 0.26667, 0.23529,  ..., 0.27843, 0.31765, 0.32549],
          [0.26275, 0.21176, 0.28627,  ..., 0.09412, 0.12941, 0.15294],
          ...,
          [0.01176, 0.01176, 0.01176,  ..., 0.18039, 0.12549, 0.07451],
          [0.01176, 0.01176, 0.01176,  ..., 0.14902, 0.09020, 0.03529],
          [0.01176, 0.01176, 0.01176,  ..., 0.12549, 0.06275, 0.00784]],

         [[0.19608, 0.13333, 0.11373,  ..., 0.89020, 0.90980, 0.89

In [ ]:
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.onnx_exec import execute_onnx

In [10]:
%cd ../
model_file_path = "./runs/train/exp/weights/best.qonnx.onnx"
qonnx_cleanup(model_file_path, out_file=model_file_path)

/home/su_kark/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/home/su_kark/Documents/Master Research Project/quantized-yolov5


In [11]:
input_dict = {"global_in": test_input_np.astype(np.float32) / 255.0}

In [12]:
model = ModelWrapper(model_file_path)
output_dict = execute_onnx(model, input_dict, return_full_exec_context=True)

In [13]:
print("OUTPUT KEYS:", output_dict.keys())

OUTPUT KEYS: dict_keys(['global_in', 'global_out', 'Quant_0_out0', 'Quant_10_out0', 'Quant_1_out0', 'Quant_11_out0', 'Quant_2_out0', 'Quant_12_out0', 'Quant_3_out0', 'Quant_13_out0', 'Quant_4_out0', 'Quant_14_out0', 'Quant_5_out0', 'Quant_15_out0', 'Quant_6_out0', 'Quant_16_out0', 'Quant_7_out0', 'Quant_17_out0', 'Quant_8_out0', 'Quant_18_out0', 'Quant_9_out0', 'Quant_0_param0', 'Quant_0_param1', 'Quant_0_param2', 'Quant_0_param3', 'Quant_10_param0', 'Quant_1_param3', 'Quant_1_param0', 'Quant_1_param1', 'Quant_2_param0', 'Quant_2_param1', 'Quant_3_param0', 'Quant_3_param1', 'Quant_4_param0', 'Quant_4_param1', 'Quant_5_param0', 'Quant_5_param1', 'Quant_6_param0', 'Quant_6_param1', 'Quant_7_param0', 'Quant_7_param1', 'Quant_8_param0', 'Quant_8_param1', 'Quant_9_param0', 'Quant_9_param1', 'Div_0_param0', 'Quant_19_param0', 'Conv_0_out0', 'BatchNormalization_0_out0', 'Relu_0_out0', 'MaxPool_0_out0', 'Conv_1_out0', 'BatchNormalization_1_out0', 'Relu_1_out0', 'MaxPool_1_out0', 'Conv_2_out0',

In [14]:
print(output_dict['global_out'].shape)

(1, 18, 13, 13)


In [ ]:
import torch

from finn.util.basic import make_build_dir
from finn.util.visualization import showInNetron
from finn.util.pytorch import ToTensor
from brevitas.export import export_qonnx
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs

build_dir = "./"

## Tidyup Transformation

In [ ]:
model = ModelWrapper(model_file_path)
model = model.transform(ConvertQONNXtoFINN())
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(RemoveStaticGraphInputs())
model.save(build_dir + "/yolov1-tiny_tidy.onnx")

In [ ]:
showInNetron(build_dir+"/yolov1-tiny_tidy.onnx")

In [ ]:
model = ModelWrapper(build_dir+"/yolov1-tiny_tidy.onnx")
global_inp_name = model.graph.input[0].name
ishape = model.get_tensor_shape(global_inp_name)

# preprocessing: torchvision's ToTensor divides uint8 inputs by 255
totensor_pyt = ToTensor()
chkpt_preproc_name = build_dir+"/yolov1-tiny_preproc.onnx"
export_qonnx(totensor_pyt, torch.randn(ishape), chkpt_preproc_name)
qonnx_cleanup(chkpt_preproc_name, out_file=chkpt_preproc_name)
pre_model = ModelWrapper(chkpt_preproc_name)
pre_model = pre_model.transform(ConvertQONNXtoFINN())

# join preprocessing and core model
model = model.transform(MergeONNXModels(pre_model))
# add input quantization annotation: UINT8 for all BNN-PYNQ models
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

In [ ]:
from qonnx.transformation.infer_datatypes import InferDataTypes
chkpt_name = build_dir+"/yolov1-tiny_preproc_tidy.onnx"

# tidy-up again
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
model.save(chkpt_name)

In [ ]:
showInNetron(build_dir+"/yolov1-tiny_preproc_tidy.onnx")

## Streamlining

In [ ]:
from finn.transformation.streamline import Streamline
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.bipolar_to_xnor import ConvertBipolarMatMulToXnorPopcount
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC, MoveScalarLinearPastInvariants
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors

In [ ]:
model = ModelWrapper(build_dir + "/yolov1-tiny_preproc_tidy.onnx")

In [ ]:
model = model.transform(MoveScalarLinearPastInvariants())
model = model.transform(Streamline())
model = model.transform(LowerConvsToMatMul())
model = model.transform(MakeMaxPoolNHWC())
model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())
model = model.transform(ConvertBipolarMatMulToXnorPopcount())
model = model.transform(Streamline())
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())
model.save(build_dir + "/yolov1-tiny_streamlined.onnx")

In [ ]:
showInNetron(build_dir+"/yolov1-tiny_streamlined.onnx")

In [ ]:
from finn.util.basic import pynq_part_map

# change this if you have a different PYNQ board, see list above
pynq_board = "Pynq-Z2"
fpga_part = pynq_part_map[pynq_board]
target_clk_ns = 10

## Conversion to HW layers

In [ ]:
import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.create_dataflow_partition import (
    CreateDataflowPartition,
)
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.infer_data_layouts import InferDataLayouts

In [ ]:
model = ModelWrapper(build_dir + "/yolov1-tiny_streamlined.onnx")
model = model.transform(to_hw.InferBinaryMatrixVectorActivation())
model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
model = model.transform(to_hw.InferThresholdingLayer())
model = model.transform(to_hw.InferConvInpGen())
model = model.transform(to_hw.InferStreamingMaxPool())
model = model.transform(absorb.AbsorbConsecutiveTransposes())
model = model.transform(RemoveCNVtoFCFlatten())
parent_model = model.transform(CreateDataflowPartition())

parent_model.save(build_dir + "/yolov1-tiny_dataflow_parent.onnx")
sdp_node = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")[0]
sdp_node = getCustomOp(sdp_node)
dataflow_model_filename = sdp_node.get_nodeattr("model")
# save the dataflow partition with a different name for easier access
# and specialize the layers to HLS variants
dataflow_model = ModelWrapper(dataflow_model_filename)
dataflow_model = dataflow_model.transform(SpecializeLayers(fpga_part))
dataflow_model.save(build_dir + "/yolov1-tiny_dataflow_model.onnx")

In [ ]:
showInNetron(build_dir + "/yolov1-tiny_dataflow_parent.onnx")

In [ ]:
showInNetron(build_dir + "/yolov1-tiny_dataflow_model.onnx")

## Folding: Adjusting the Parallelism

In [ ]:
model = ModelWrapper(build_dir + "/yolov1-tiny_dataflow_model.onnx")
fc_layers = model.get_nodes_by_op_type("MVAU_hls")

In [ ]:
# each tuple is (PE, SIMD, in_fifo_depth) for a layer
folding = [
    (8, 9, [8]),
    (4, 4, [8]),
    (4, 4, [8]),
    (2, 2, [8]),
    (2, 2, [8]),
    (2, 2, [4]),
    (4, 4, [4]),
    (4, 4, [8]),
    (4, 4, [8]),
    (1, 8, [8]),
]

for fcl, (pe, simd, ififodepth) in zip(fc_layers, folding):
    fcl_inst = getCustomOp(fcl)
    fcl_inst.set_nodeattr("PE", pe)
    fcl_inst.set_nodeattr("SIMD", simd)
    fcl_inst.set_nodeattr("inFIFODepths", ififodepth)

In [ ]:
fc_layers = model.get_nodes_by_op_type("ConvolutionInputGenerator_rtl")

In [ ]:
folding = [
    (3),
    (2),
    (2),
    (2),
    (2),
    (2),
    (2),
    (2),
    (2),
]
for fcl, (simd) in zip(fc_layers, folding):
    fcl_inst = getCustomOp(fcl)
    fcl_inst.set_nodeattr("SIMD", simd)

In [ ]:
model = model.transform(GiveUniqueNodeNames())
model.save(build_dir + "/yolov1-tiny_folded.onnx")

In [ ]:
showInNetron(build_dir + "/yolov1-tiny_folded.onnx")

In [ ]:
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild

model = ModelWrapper(build_dir+"/yolov1-tiny_folded.onnx")
model = model.transform(ZynqBuild(platform = pynq_board, period_ns = target_clk_ns))

In [ ]:
from finn.transformation.fpgadataflow.make_pynq_driver import MakePYNQDriver
model = model.transform(MakePYNQDriver("zynq-iodma"))

In [ ]:
model.save(build_dir + "/yolov1-tiny_synth.onnx")
showInNetron(build_dir + "/yolov1-tiny_synth.onnx")

In [ ]:
model = ModelWrapper(build_dir + "/yolov1-tiny_synth.onnx")
model.model.metadata_props

In [ ]:
showInNetron(build_dir + "/yolov1-tiny_synth.onnx", port=2222)

In [ ]:
! ls {model.get_metadata_prop("vivado_pynq_proj")}